# BlueSpotter — Cellpose-SAM training (Colab)

Fine-tunes Cellpose-SAM on locus coeruleus data. **Code** comes from GitHub, **data + model** live in Google Drive, and each run is tracked with **MLflow**.

**Before running:** Runtime → Change runtime type → **GPU**.

## 1. Setup — mount Drive, clone repo, install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone (or update) the code repo. Private repo -> you'll be prompted, or use a
# fine-grained token. Data/models are NOT here; they live in Drive.
import os
REPO = '/content/BlueSpotter'
if not os.path.exists(REPO):
    !git clone https://github.com/TeamPrigge/BlueSpotter.git {REPO}
else:
    !cd {REPO} && git pull --ff-only
%cd {REPO}

In [ ]:
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, '/content/BlueSpotter/src')

## 2. Check config

Confirm `params.yaml` points at the right Drive folder. Edit `drive.root` there if BlueSpotter lives in a Shared drive.

In [ ]:
from bluespotter.config import load_config
cfg = load_config()
print('Drive root :', cfg.drive_root)
print('Data dir   :', cfg.data_dir, '(exists:', cfg.data_dir.exists(), ')')
print('Model dir  :', cfg.model_dir)
print('Tracking   :', cfg.tracking_uri())

## 3. Train

Caches data Drive→local SSD, fine-tunes Cellpose-SAM, logs to MLflow, writes the model back to Drive. Hyperparameters come from `params.yaml`.

In [ ]:
from bluespotter.train import run
model_path = run(cfg)
print('Done. Model at:', model_path)

## 4. (Optional) Browse MLflow runs

In [ ]:
# Point mlflow ui at the Drive-synced store and open via Colab proxy.
uri = cfg.tracking_uri().replace('file://', '')
print('backend-store-uri:', uri)
# !mlflow ui --backend-store-uri "file://{uri}" --port 5000 &